## Imports & Dependency Checks
We need to install the competition wheels and a custom wheels for `vllm` as well.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import pandas as pd
import os
import textwrap
import time

# Record start timestamp to avoid missing the 15min first action deadline
NOTEBOOK_START_TIME_S = time.time()
notebook_start_path = Path("/kaggle/working/notebook_start_time_s.txt")
notebook_start_path.parent.mkdir(parents=True, exist_ok=True)
notebook_start_path.write_text(str(NOTEBOOK_START_TIME_S))
print("NOTEBOOK_START_TIME_S =", NOTEBOOK_START_TIME_S)

# Scan all inputs for vLLM wheels
vllm_wheels = list(Path("/kaggle/input").rglob("vllm*.whl"))
print(vllm_wheels[:5])

if not vllm_wheels:
  raise FileNotFoundError("No vllm wheel found under /kaggle/input")

WHEEL_DIR = vllm_wheels[0].parent
print("WHEEL_DIR =", WHEEL_DIR)

def ensure_vllm():
  if importlib.util.find_spec("vllm") is not None:
      print("vllm is already available, skipping install from wheel!")
      return

  if not WHEEL_DIR.exists():
      raise FileNotFoundError(f"vLLM is not installed and wheel dir is missing: {WHEEL_DIR}")

  subprocess.check_call([
      sys.executable,
      "-m",
      "pip",
      "install",
      "--no-index",
      "--find-links",
      str(WHEEL_DIR),
      "vllm==0.19.1",
      "openai==2.24.0",
      "openai-harmony==0.0.8",
  ])

ensure_vllm()

Install ARC-AGI3 competition wheels

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
import vllm
import arc_agi

print(f"vllm: {vllm.__version__}")
print(f"arc_agi available")

## Create an Agent file 
We subclass `LLM`, which provides a default integration with the ARC-AGI3 game server.
Custom implementations are provided for `choose_action`, `is_done` and `cleanup`, which handle action selection, agent state and post-game cleanup with extended logging.
For model inference, we utilize the `openai` SDK's `OpenAICLient`, which can communicate directly with the `vllm` server's API, as both support the OpenAI communication protocol.
The model uses structured output to select one of the current valid actions in JSON format, given the game state.
The agent will be written to a file `kaggle/working/my_agent.py`, which is then used by the ARC-AGI agent framework to deploy it later on.

### Configuring server & agent behaviour
The code provides a few constants and class members that control the behaviour of `MyAgent`.
All of them are written in UPPERCASE and can be edited.
Comments next to each of them describe which can/should be changed and what aspect of the agent they control.

In [ ]:
%%writefile /kaggle/working/my_agent.py

import json
import logging
import os
import re
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from typing import Any
from urllib.error import URLError
from urllib.request import urlopen

import openai
from openai import OpenAI as OpenAIClient

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from PIL import ImageDraw

if not hasattr(ImageDraw, "Coords"):
    ImageDraw.Coords = Any

try:
    from .llm_agents import LLM
except ImportError:
    from agents.templates.llm_agents import LLM

from arcengine import FrameData, GameAction

# Agent Version for Local Experiment Versioning
MY_AGENT_VERSION = "gpt-oss-json-guided-no-reset-v2"

# Model Input
# -----------
# Change this when using a different model after attaching it as input to the notebook.
# -> Take care to also include any tokenizer downloads that you may need, the current notebook only preloads tiktoken
# cl100k_base and o200k_base for gpt-oss-120b (see vllm-deps input notebook).
AGENT_MODEL_NAME = "gpt-oss-120b"
MODEL_ID = "/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1"
VLLM_MODEL_PATH_ENV = "VLLM_MODEL_PATH"

# Server Startup Configuration
# ----------------------------
# Default parameters that configure the vllm-server networking and API
VLLM_HOST = "127.0.0.1"
VLLM_PORT = 8000
OPENAI_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
HEALTH_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/health"
MODELS_URL = f"{OPENAI_BASE_URL}/models"
OPENAI_API_KEY = "EMPTY" # Dummy key
# Timeout for the vLLM server, after which it will fail the run.
VLLM_STARTUP_TIMEOUT_SEC = 1800
# Polling rate for the vLLM server start health check.
VLLM_POLL_INTERVAL_SEC = 5.0

# vLLM Engine Parameters & Model Sampling Parameters
# (for further configuration, add flags to the command built in `build_vllm_command`).
# --------------------------------------------------
# Controls the maximum number of parallel decoding sequences in vLLM.
# Only relevant for parallel inference, which the current notebook doesn't do.
MAX_NUM_SEQS = 12 
# Controls the maximum context length of the model. Increasing this will decrease 
# performance (larger forward pass), but will allow the model more room to reason.
# Can fill up quickly if you pass many previous states into context!
# gpt-oss-120b has a maximum of 128k tokens of context.
MAX_MODEL_LEN = 64000
# Key-Value Cache quantization. We use 'fp8' to conserve precious GPU memory.
KV_CACHE_DTYPE = "fp8"
# We only have 1 GPU, so we can't parallelize.
TENSOR_PARALLEL_SIZE = 1
# Disables/Enables the CUDA-Graph compilation (eager=no CUDA graph compilation).
# Enable this to increase throughput, at the cost of more startup time for compilation.
ENFORCE_EAGER = True


vllm_process: subprocess.Popen | None = None
TIKTOKEN_ENCODINGS_DIR = None


def start_vllm_server(timeout_s: int = VLLM_STARTUP_TIMEOUT_SEC) -> subprocess.Popen:
    """Start or reuse the local vLLM server and block until it is healthy."""
    global vllm_process

    configure_openai_env(include_tiktoken=True)

    if _server_process_is_ready(vllm_process):
        print(f"vLLM is already running at {OPENAI_BASE_URL}")
        return vllm_process  # type: ignore[return-value]

    if vllm_process is not None and vllm_process.poll() is None:
        print(f"Existing vLLM process pid={vllm_process.pid} is not ready; restarting.")
        stop_vllm_server()

    cmd = build_vllm_command()
    print("\n--- starting vLLM ---")
    print(" ".join(cmd))

    vllm_process = subprocess.Popen(
        cmd,
        stdout=None,
        stderr=subprocess.STDOUT,
        env=build_vllm_env(),
        text=True,
    )
    print(f"Started vLLM pid={vllm_process.pid}")

    wait_for_vllm_server(vllm_process, timeout_s=timeout_s)
    return vllm_process


def stop_vllm_server(timeout_s: int = 30) -> None:
    """Terminate the managed vLLM subprocess if this agent started one."""
    global vllm_process

    if vllm_process is None or vllm_process.poll() is not None:
        vllm_process = None
        return

    print(f"Stopping vLLM pid={vllm_process.pid}")
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        print(f"vLLM pid={vllm_process.pid} did not stop cleanly; killing.")
        vllm_process.kill()
        vllm_process.wait()
    vllm_process = None


def configure_openai_env(include_tiktoken: bool = False) -> None:
    """Point OpenAI-compatible clients and tiktoken at the local Kaggle assets."""
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    if include_tiktoken:
        os.environ["TIKTOKEN_ENCODINGS_BASE"] = str(get_tiktoken_encodings_dir())


def build_vllm_command(
    model_id: str | None = None,
    served_model_name: str = AGENT_MODEL_NAME,
) -> list[str]:
    """Build the vLLM server command used by the notebook and agent startup."""
    model_id = model_id or os.getenv(VLLM_MODEL_PATH_ENV) or MODEL_ID
    # Can add more configuration to the vLLM server command here:
    args = [
        "--served-model-name",
        served_model_name,
        "--host",
        VLLM_HOST,
        "--port",
        str(VLLM_PORT),
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "openai",
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--kv-cache-dtype",
        KV_CACHE_DTYPE,
        "--tensor-parallel-size",
        str(TENSOR_PARALLEL_SIZE),
    ]
    if ENFORCE_EAGER:
        args.append("--enforce-eager")

    vllm_exe = shutil.which("vllm")
    if vllm_exe:
        return [vllm_exe, "serve", model_id, *args]

    return [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        model_id,
        *args,
    ]


def build_vllm_env() -> dict[str, str]:
    """Create the environment passed to the vLLM subprocess."""
    env = os.environ.copy()
    env["OPENAI_API_KEY"] = OPENAI_API_KEY
    env["OPENAI_BASE_URL"] = OPENAI_BASE_URL
    env["TIKTOKEN_ENCODINGS_BASE"] = str(get_tiktoken_encodings_dir())
    env.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
    return env


def wait_for_vllm_server(
    process: subprocess.Popen,
    timeout_s: int = VLLM_STARTUP_TIMEOUT_SEC,
    poll_interval_s: float = VLLM_POLL_INTERVAL_SEC,
) -> None:
    """Wait until vLLM responds to health/model checks or the process exits."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                f"vLLM exited with code {process.returncode}. "
                "Check the output above for the vLLM error log."
            )
        if server_ready(timeout=2.0):
            print(f"vLLM is ready at {OPENAI_BASE_URL}")
            return
        time.sleep(poll_interval_s)

    raise TimeoutError(
        f"Timed out waiting for vLLM after {timeout_s}s. "
        "Check the output above for the vLLM startup log."
    )


def server_ready(timeout: float = 2.0) -> bool:
    """Return True once either vLLM health or model listing succeeds."""
    return _url_ok(HEALTH_URL, timeout=timeout) or _url_ok(MODELS_URL, timeout=timeout)


def get_tiktoken_encodings_dir() -> Path:
    """Locate cached tiktoken encoding files bundled as Kaggle inputs."""
    global TIKTOKEN_ENCODINGS_DIR
    if TIKTOKEN_ENCODINGS_DIR is not None:
        return TIKTOKEN_ENCODINGS_DIR

    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not root.exists():
            continue
        for candidate_dir in root.rglob("*"):
            if not candidate_dir.is_dir():
                continue
            if (
                (candidate_dir / "cl100k_base.tiktoken").exists()
                and (candidate_dir / "o200k_base.tiktoken").exists()
            ):
                TIKTOKEN_ENCODINGS_DIR = candidate_dir
                print("TIKTOKEN_ENCODINGS_DIR =", candidate_dir)
                return candidate_dir

    raise FileNotFoundError(
        "Could not find a directory containing both "
        "cl100k_base.tiktoken and o200k_base.tiktoken."
    )


def _server_process_is_ready(process: subprocess.Popen | None) -> bool:
    return process is not None and process.poll() is None and server_ready(timeout=2.0)


def _url_ok(url: str, timeout: float = 2.0) -> bool:
    try:
        with urlopen(url, timeout=timeout) as response:
            return response.status == 200
    except (OSError, URLError):
        return False

# ----------------- Agent Definition ------------------------------

class MyAgent(LLM):
    """ARC LLM agent backed by a local vLLM gpt-oss-120b server."""

    # Default parameters, no need to change
    # Configures the name of the agent in the logs, no need to change this.
    MODEL = AGENT_MODEL_NAME
    MODEL_REQUIRES_TOOLS = False

    # Enabling observations adds an extra completion before the 
    # structured action selection
    # -> Be careful to enable this, adds to the message limit and increases 
    # token cost per action.
    DO_OBSERVATION = False
    # Token limit for the free-form observation generation.
    OBSERVATION_MAX_TOKENS = 512

    # Maximum number of actions to take per game.
    MAX_ACTIONS = 50
    # Maximum previous messages to keep in context for the current game state.
    MESSAGE_LIMIT = 2
    # How often to retry structured action selection if the model fails to respond 
    # in the specified format.
    ACTION_RETRY_LIMIT = 1
    # Maximum tokens per action selection for a game-state.
    # More tokens allow the model to reason longer, but will fill context.
    # If you allow many previous states in context, it will quickly fill up beyond the default 64k!
    ACTION_MAX_TOKENS = 8192
    # How long to wait for the vllm-server to respond per completion.
    # Increase if you expect long reasoning time per action, due to large context and ACTION_MAX_TOKENS.
    LLM_REQUEST_TIMEOUT_S = 120

    # Time-limit per ARC-AGI game in seconds.
    # -> The agent will return True for `is_done` after this time limit is reached.
    GAME_TIME_LIMIT_S = 60 * 20 
    # Time-limit by which the agent has to return its first action to comply with Kaggle's submission guidelines.
    FIRST_ACTION_DEADLINE_S = 14 * 60

    # PROMPTS:
    # 1. Game-State Representation: Adjust the prompt generated by `build_state_prompt`
    # 2. Action-Selection: Adjust the prompt generated by `build_user_prompt`


    ACTION_NAMES = [
        GameAction.RESET.name,
        GameAction.ACTION1.name,
        GameAction.ACTION2.name,
        GameAction.ACTION3.name,
        GameAction.ACTION4.name,
        GameAction.ACTION5.name,
        GameAction.ACTION6.name,
        GameAction.ACTION7.name,
    ]
    DEFAULT_ACTION_NAMES = ACTION_NAMES[1:]
    AGENT_VERSION = MY_AGENT_VERSION

    _vllm_started = False
    NOTEBOOK_START_TIME_S = 0
    NOTEBOOK_START_TIME_PATH = Path("/kaggle/working/notebook_start_time_s.txt")

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """Initialize cheaply so the first RESET action is never blocked by vLLM."""
        print(f"MyAgent version: {self.AGENT_VERSION}")
        configure_openai_env()
        super().__init__(*args, **kwargs)
        self._deadline_hit = False
        self._game_started_monotonic = 0.0

    @property
    def startup_elapsed_s(self) -> float:
        start_time_s = float(self.NOTEBOOK_START_TIME_S)
        if start_time_s <= 0 and self.NOTEBOOK_START_TIME_PATH.exists():
            try:
                start_time_s = float(self.NOTEBOOK_START_TIME_PATH.read_text().strip())
            except (OSError, ValueError):
                start_time_s = 0.0
        if start_time_s <= 0:
            return self.game_elapsed_s
        return max(0.0, time.time() - start_time_s)

    @property
    def game_elapsed_s(self) -> float:
        if self._game_started_monotonic:
            return time.monotonic() - self._game_started_monotonic
        if self.timer:
            return time.time() - self.timer
        return 0.0

    @property
    def game_time_remaining_s(self) -> float | None:
        limit = max(0, int(self.GAME_TIME_LIMIT_S))
        if limit <= 0:
            return None
        return max(0.0, limit - self.game_elapsed_s)

    def _time_limit_expired(self) -> bool:
        remaining = self.game_time_remaining_s
        return remaining is not None and remaining <= 0

    def _mark_deadline_hit(self, reason: str) -> None:
        if not self._deadline_hit:
            logging.info(
                "%s; stopping %s after %s actions and %.2f seconds",
                reason,
                self.game_id,
                self.action_counter,
                self.game_elapsed_s,
            )
        self._deadline_hit = True

    def _ensure_vllm_ready(self) -> bool:
        if self.__class__._vllm_started:
            configure_openai_env()
            return True

        remaining = self.game_time_remaining_s
        if remaining is not None and remaining <= 0:
            self._mark_deadline_hit("Per-game time limit reached before vLLM startup")
            return False

        timeout_s = int(remaining) if remaining is not None else VLLM_STARTUP_TIMEOUT_SEC
        timeout_s = max(1, timeout_s)
        try:
            start_vllm_server(timeout_s=timeout_s)
        except TimeoutError:
            self._mark_deadline_hit("Per-game time limit reached while waiting for vLLM")
            return False
        except Exception as e:
            logging.exception("vLLM startup failed: %s", e)
            return False

        self.__class__._vllm_started = True
        configure_openai_env()
        return True

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Use the parent game-state check, plus a wall-clock stop before new work."""
        if not self._game_started_monotonic:
            self._game_started_monotonic = time.monotonic()
        if super().is_done(frames, latest_frame):
            return True
        if self._deadline_hit:
            return True
        if self.action_counter == 0:
            return False
        if self._time_limit_expired():
            self._mark_deadline_hit("Per-game time limit reached before next action")
            return True
        return False

    def choose_action(
        self,
        frames: list[FrameData],
        latest_frame: FrameData,
    ) -> GameAction:
        """Choose the next ARC action using guided JSON for simplicity."""
        self._quiet_openai_logging()
        available_action_names = self._available_action_names(latest_frame)

        if self.action_counter == 0:
            first_action_deadline_s = max(1, int(self.FIRST_ACTION_DEADLINE_S))
            startup_elapsed_s = self.startup_elapsed_s
            if startup_elapsed_s > first_action_deadline_s:
                logging.warning(
                    f"First action for {self.game_id} is being selected after {startup_elapsed_s:.2f} seconds since notebook start, past the {first_action_deadline_s:.2f} second target",
                )
            return self._initial_reset_action()

        if self._time_limit_expired():
            self._mark_deadline_hit("Per-game time limit reached before LLM action")
            return self._fallback_action(available_action_names)

        if not self._ensure_vllm_ready():
            return self._fallback_action(available_action_names)

        try:
            return self._choose_llm_action(latest_frame, available_action_names)
        except Exception as e:
            logging.exception("Action selection failed; falling back to %s: %s", available_action_names[0], e)
            return self._fallback_action(available_action_names)

    def _fallback_action(self, available_action_names: list[str]) -> GameAction:
        action_name = available_action_names[0] if available_action_names else GameAction.ACTION5.name
        if action_name == GameAction.RESET.name:
            action_name = GameAction.ACTION5.name
        return self._game_action_from_data({"action": action_name})

    def _choose_llm_action(
        self,
        latest_frame: FrameData,
        available_action_names: list[str],
    ) -> GameAction:
        client = OpenAIClient(
            base_url=OPENAI_BASE_URL,
            api_key=OPENAI_API_KEY,
            timeout=max(1, int(self.LLM_REQUEST_TIMEOUT_S)),
        )
        self._prepare_action_messages(latest_frame, available_action_names)

        if self.DO_OBSERVATION:
            self._request_observation(client)
            self._append_action_prompt(latest_frame, available_action_names)

        data = self._request_json_action(client, available_action_names)
        data = self._normalize_action_data(data, available_action_names)
        self._record_assistant_action(data)
        return self._game_action_from_data(data)

    def _initial_reset_action(self) -> GameAction:
        data = {"action": GameAction.RESET.name}
        self.messages = [{"role": "assistant", "content": json.dumps(data)}]
        return GameAction.RESET

    def take_action(self, action: GameAction) -> FrameData | None:
        """Submit an action without letting gateway/runtime errors kill the worker thread."""
        try:
            return super().take_action(action)
        except Exception as e:
            logging.exception("Action submission failed for %s; stopping game: %s", action.name, e)
            self._mark_deadline_hit("Action submission failed")
            return None

    def _prepare_action_messages(
        self,
        latest_frame: FrameData,
        available_action_names: list[str],
    ) -> None:
        self.messages = [{"role": "user", "content": self.build_state_prompt(latest_frame)}]
        self._append_action_prompt(latest_frame, available_action_names)

    def _append_action_prompt(
        self,
        latest_frame: FrameData,
        available_action_names: list[str],
    ) -> None:
        self.messages.append(
            {
                "role": "user",
                "content": self.build_user_prompt(
                    latest_frame,
                    available_action_names=available_action_names,
                ),
            }
        )
        self.messages = self.messages[-self.MESSAGE_LIMIT :]

    def _request_json_action(
        self,
        client: OpenAIClient,
        available_action_names: list[str],
    ) -> dict[str, Any]:
        """Ask vLLM for one JSON action constrained to currently available actions."""
        retry_prompt = self._retry_prompt(available_action_names)
        schema = self._action_json_schema(available_action_names)

        for attempt in range(self.ACTION_RETRY_LIMIT + 1):
            request_messages = self.messages if attempt == 0 else [*self.messages, retry_prompt]
            response = self._create_action_completion(client, request_messages, schema, attempt)
            if response is None:
                continue

            choices = getattr(response, "choices", None) or []
            if not choices:
                logging.warning("vLLM JSON action response had no choices on attempt %s/%s", attempt + 1, self.ACTION_RETRY_LIMIT + 1)
                continue

            message = choices[0].message
            data = self._parse_json_response(self._message_content(message))
            if data is not None:
                return data

            logging.warning(
                "JSON action response was invalid on attempt %s/%s; message=%r",
                attempt + 1,
                self.ACTION_RETRY_LIMIT + 1,
                message.model_dump(),
            )

        return {"action": GameAction.ACTION5.name}

    def _create_action_completion(
        self,
        client: OpenAIClient,
        request_messages: list[dict[str, Any]],
        schema: dict[str, Any],
        attempt: int,
    ) -> Any | None:
        """Submit one guided-JSON chat completion and account for token usage."""
        try:
            response = client.chat.completions.create(
                model=self.MODEL,
                messages=request_messages,
                max_tokens=self.ACTION_MAX_TOKENS,
                extra_body={"guided_json": schema},
            )
        except openai.OpenAIError as e:
            logging.warning(
                f"vLLM JSON action request failed on attempt {attempt + 1}/{self.ACTION_RETRY_LIMIT + 1}: {e}",
            )
            return None
        except Exception as e:
            logging.exception(
                "Unexpected vLLM JSON action request failure on attempt %s/%s: %s",
                attempt + 1,
                self.ACTION_RETRY_LIMIT + 1,
                e,
            )
            return None

        usage = getattr(response, "usage", None)
        if usage and getattr(usage, "total_tokens", None):
            self.track_tokens(usage.total_tokens)
        return response

    def _request_observation(self, client: OpenAIClient) -> None:
        """Optionally add a short natural-language observation before action choice."""
        try:
            response = client.chat.completions.create(
                model=self.MODEL,
                messages=self.messages,
                max_tokens=self.OBSERVATION_MAX_TOKENS,
            )
        except openai.OpenAIError as e:
            logging.warning("vLLM observation request failed: %s", e)
            return
        except Exception as e:
            logging.exception("Unexpected vLLM observation request failure: %s", e)
            return

        choices = getattr(response, "choices", None) or []
        if not choices:
            logging.warning("vLLM observation response had no choices")
            return

        content = self._message_content(choices[0].message)
        usage = getattr(response, "usage", None)
        if usage and getattr(usage, "total_tokens", None):
            self.track_tokens(usage.total_tokens, content)
        self.messages.append({"role": "assistant", "content": content})
        self.messages = self.messages[-self.MESSAGE_LIMIT :]

    def _available_action_names(self, latest_frame: FrameData) -> list[str]:
        """Map environment-provided action ids to names, excluding RESET after turn zero."""
        names: list[str] = []
        for action_id in getattr(latest_frame, "available_actions", None) or []:
            try:
                name = GameAction(action_id).name
            except ValueError:
                continue
            if name != GameAction.RESET.name and name in self.ACTION_NAMES:
                names.append(name)
        return names or list(self.DEFAULT_ACTION_NAMES)

    def _action_json_schema(self, available_action_names: list[str]) -> dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "action": {"type": "string", "enum": available_action_names},
                "x": {"type": "integer", "minimum": 0, "maximum": 63},
                "y": {"type": "integer", "minimum": 0, "maximum": 63},
            },
            "required": ["action"],
            "additionalProperties": False,
        }

    def _retry_prompt(self, available_action_names: list[str]) -> dict[str, str]:
        return {
            "role": "user",
            "content": (
                "Your previous response was not valid action JSON. Return only one JSON object now. "
                f"The action must be one of: {', '.join(available_action_names)}. "
                "Do not include prose, markdown, code fences, or commentary."
            ),
        }

    def _normalize_action_data(
        self,
        data: dict[str, Any],
        available_action_names: list[str],
    ) -> dict[str, Any]:
        """Coerce model JSON into a valid action allowed by the current frame."""
        action_name = str(data.get("action", GameAction.ACTION5.name)).upper()
        if action_name not in available_action_names:
            logging.warning(
                "Invalid or unavailable ARC action %r; defaulting to %s",
                action_name,
                available_action_names[0],
            )
            action_name = available_action_names[0]

        normalized = {"action": action_name}
        if action_name == GameAction.ACTION6.name:
            normalized["x"] = self._clamp_coordinate(data.get("x", 0))
            normalized["y"] = self._clamp_coordinate(data.get("y", 0))
        return normalized

    def _record_assistant_action(self, data: dict[str, Any]) -> None:
        self.messages = [*self.messages[-2:], {"role": "assistant", "content": json.dumps(data)}]

    def _game_action_from_data(self, data: dict[str, Any]) -> GameAction:
        """Convert normalized JSON into the ARC engine GameAction object."""
        action_name = str(data.get("action", GameAction.ACTION5.name)).upper()
        action = GameAction.from_name(action_name)
        if action_name == GameAction.ACTION6.name:
            action.set_data(
                {
                    "x": self._clamp_coordinate(data.get("x", 0)),
                    "y": self._clamp_coordinate(data.get("y", 0)),
                }
            )
        else:
            action.set_data({})
        return action

    def _message_content(self, message: Any) -> str:
        """Read vLLM output from the Pydantic v2 message payload."""
        data = message.model_dump()
        for key in ["content", "reasoning_content", "reasoning"]:
            value = data.get(key)
            if value:
                return str(value)
        return ""

    def _parse_json_response(self, content: str) -> dict[str, Any] | None:
        """Parse strict JSON first, then recover a single JSON object from text."""
        if not content:
            return None

        data = self._loads_json_object(content)
        if data is not None:
            return data

        match = re.search(r"\{.*\}", content, flags=re.DOTALL)
        if not match:
            return None
        return self._loads_json_object(match.group(0))

    def _loads_json_object(self, content: str) -> dict[str, Any] | None:
        """Load one JSON object, returning None for invalid JSON or non-objects."""
        try:
            data = json.loads(content)
        except json.JSONDecodeError:
            return None
        return data if isinstance(data, dict) else None

    def _clamp_coordinate(self, value: Any) -> int:
        try:
            coordinate = int(value)
        except (TypeError, ValueError):
            coordinate = 0
        return min(63, max(0, coordinate))

    def _quiet_openai_logging(self) -> None:
        logging.getLogger("openai").setLevel(logging.CRITICAL)
        logging.getLogger("httpx").setLevel(logging.CRITICAL)

    def build_state_prompt(self, latest_frame: FrameData) -> str:
        """Render the latest ARC frame and score as the model's current state."""
        return textwrap.dedent(
            """
            # RESULT OF PREVIOUS ACTION
            State: {state}
            Score: {score}

            # FRAME
            {latest_frame}
            """.format(
                latest_frame=self.pretty_print_3d(latest_frame.frame),
                score=latest_frame.levels_completed,
                state=latest_frame.state.name,
            )
        ).strip()

    def build_user_prompt(
        self,
        latest_frame: FrameData,
        available_action_names: list[str] | None = None,
    ) -> str:
        """Build the action-selection instruction shown after the current frame."""
        available_action_names = available_action_names or self.DEFAULT_ACTION_NAMES
        return textwrap.dedent(
            f"""
            # CONTEXT
            You are playing an ARC-AGI-3 dynamic game. Your objective is to WIN
            and avoid GAME_OVER while minimizing the number of actions.

            Each action produces one Frame. A Frame contains one or more Grids.
            Each Grid is a matrix with dimensions INT<0,63> by INT<0,63> and
            cell values INT<0,15>.

            # AVAILABLE ACTIONS THIS TURN
            {', '.join(available_action_names)}
            ACTION6 is a click action and should include integer x and y
            coordinates from 0 to 63.

            # RESPONSE FORMAT
            Return only one JSON object. Do not include prose, markdown, code
            fences, or commentary.

            Valid examples:
            {{"action":"ACTION1"}}
            {{"action":"ACTION6","x":0,"y":0}}
            """
        ).strip()

    def cleanup(self, *args: Any, **kwargs: Any) -> None:
        """Record concise JSON-mode metadata, then delegate to ARC cleanup."""
        if self._cleanup and hasattr(self, "recorder") and not self.is_playback:
            self.recorder.record(
                {
                    "agent_version": self.AGENT_VERSION,
                    "llm_user_prompt": self.build_user_prompt(
                        self.frames[-1],
                        available_action_names=self.DEFAULT_ACTION_NAMES,
                    ),
                    "llm_response_format": "guided_json_action",
                    "llm_available_actions": self.DEFAULT_ACTION_NAMES,
                }
            )
        super().cleanup(*args, **kwargs)

#### Setting up the `agents` library
Next, we copy the agent framework provided by the competition into the writable `/kaggle/working` directory.
The input directories themselves are not writable. Then, we can copy our custom agent into the agent templates.

In [ ]:
AGENTS_INPUT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"
AGENTS_WD = "/kaggle/working/ARC-AGI-3-Agents"
AGENTS_ENV = f"{AGENTS_WD}/.env"
TEMPLATE_INPUT = "/kaggle/working/my_agent.py"
TEMPLATE_TARGET = f"{AGENTS_WD}/agents/templates/my_agent.py"
AGENTS_INIT = f"{AGENTS_WD}/agents/__init__.py"

# Force refresh of the WD and agent template.
!rm -rf {AGENTS_WD}
!cp -r {AGENTS_INPUT} {AGENTS_WD}
!cp -f {TEMPLATE_INPUT} {TEMPLATE_TARGET}
!ls -lah {TEMPLATE_TARGET}

Then, we also need to adjust the `__init__.py` file of `agents` to avoid imports of unused libraries like `langgraph`, `smolagents`, etc.

In [ ]:
  with open(AGENTS_INIT, "w") as f:
      f.write(textwrap.dedent("""from typing import Type
  from dotenv import load_dotenv
  from .agent import Agent, Playback
  from .swarm import Swarm
  from .templates.random_agent import Random
  from .templates.my_agent import MyAgent

  load_dotenv()

  AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
      "random": Random,
      "myagent": MyAgent,
  }
  """))
      print("Updated __init__.py with the agent template")


Next, we need to setup a `.env` file containing all environment variables for the ARC-AGI3 game server.


* __Competition Submission__: For the live competition (when `KAGGLE_IS_COMPETITION_RERUN` is `True`), this will use the Kaggle-hosted game server, no local environments are required.
* __Local/Offline Session__: To test the notebook in a local Jupyter session (or one on Kaggle), we setup an _offline_ ARC-AGI server ourselves using the public input environments.

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
     with open(AGENTS_ENV, "w") as f:
      f.write(textwrap.dedent("""SCHEME=http
  HOST=gateway
  PORT=8001
  ARC_API_KEY=test-key-123
  ARC_BASE_URL=http://gateway:8001/
  OPERATION_MODE=online
  ENVIRONMENTS_DIR=
  RECORDINGS_DIR=/kaggle/working/server_recording
  """))
     print("Updated .env for the ARC-AGI competition")
else:
    print("Writing OFFLINE config to .env for local testing")
    with open(AGENTS_ENV, "w") as f:
        f.write(textwrap.dedent("""
SCHEME=http
HOST=127.0.0.1
PORT=8001
ARC_API_KEY=test-key-123
"""))
    os.environ.pop("ARC_BASE_URL", None)
    os.environ.pop("ARC_API_KEY", None)

In [ ]:
!cat {AGENTS_ENV}

### Running the Agent

When running _offline_ (not a competition submission), we spawn a local ARC-AGI server using the environment files provided via the Kaggle input.
Recordings of the games being played and server logs are placed into `/kaggle/working`.

In [ ]:
def spawn_arc_local_server():
    """Launches a local ARC-AGI game server using the environment files."""
    return subprocess.Popen(
        [
            sys.executable,
            "-c",
            textwrap.dedent("""
    import os
    from pathlib import Path
    from arc_agi import Arcade, OperationMode
    
    os.environ["OPERATION_MODE"] = "OFFLINE"
    os.environ["ENVIRONMENTS_DIR"] = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    os.environ["RECORDINGS_DIR"] = "/kaggle/working/server_recording"
    
    Path(os.environ["RECORDINGS_DIR"]).mkdir(parents=True, exist_ok=True)
    
    Arcade(operation_mode=OperationMode.OFFLINE).listen_and_serve(
        host="0.0.0.0",
        port=8001,
        competition_mode=True,
        save_all_recordings=True,
    )
    """),
        ],
        stdout=open("/kaggle/working/arc_server.log", "w"),
        stderr=subprocess.STDOUT,
    )

Finally, we can run either a competition submission, or an _offline_ one using:
* __Competition Submission__: Waits for the ARC-AGI server to start by trying to reach the `games` gateway. Upon success, spawns an instance of `myagent` that will play all games.
* __Local/Offline Submission__: Spawns a local ARC-AGI server using the provided public games, waits for it to spawn, then deploys `myagent` to play the game BP35. Adapt this to any game in `/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files` to play a different one.

In [ ]:
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("Running competition mode")
    subprocess.run(
        [
            "curl",
            "--fail",
            "--retry",
            "999",
            "--retry-all-errors",
            "--retry-delay",
            "5",
            "--retry-max-time",
            "600",
            "http://gateway:8001/api/games",
        ],
        check=True,
    )
    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"
    subprocess.run(
        [sys.executable, "main.py", "--agent", "myagent"],
        cwd=AGENTS_WD,
        check=True,
        env=env,
    )
else:
    print("Running local testing mode")
    from arc_agi import Arcade, OperationMode

    spawn_arc_local_server()
    time.sleep(5)
    print("Started local ARC server.")
    !cd {AGENTS_WD} && python main.py --agent myagent --game=bp35

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Non-competition mode: Must produce a dummy submission file to be able to submit to competition.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()